# 17 - Final Forward Decision and Submission

## Objective

This notebook documents the final forward-submission decision and prepares the final challenge files.

After all experiments, the final choice is based on:

1. Standard KFold CV results;
2. OOD test-set diagnostics;
3. controlled blend diagnostics;
4. inverse-design feasibility recheck;
5. Leave-One-Gravity-Out validation with bootstrap uncertainty.

Final decision:

```text
Forward final candidate: prediction_submission_hybrid_mlp_distance_w_0.65.csv
Inverse final candidate: design_submission.csv
```

This notebook is intentionally a final reference notebook. It does not train new models.
It loads existing candidate files, validates them, compares their distributions, loads the LOGO results from notebook 16, and can optionally copy the selected candidate to the official filename.

## 1. Imports and project paths

In [1]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
SUBMISSIONS_DIR = OUTPUTS_DIR / "submissions"
REPORTS_DIR = PROJECT_ROOT / "reports"

print("Project root:", PROJECT_ROOT)
print("Submissions:", SUBMISSIONS_DIR)
print("Reports:", REPORTS_DIR)

Project root: /home/alouiyaz/projects/boom-challenge-ejecta-prediction
Submissions: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions
Reports: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/reports


## 2. Final decision summary

The final forward candidates are:

- `prediction_submission.csv`: conservative ExtraTrees baseline;
- `prediction_submission_hybrid_mlp_distance_w_0.65.csv`: hybrid MLP-distance candidate;
- `prediction_submission_controlled_blend_alpha_0.25.csv`: conservative controlled blend;
- `prediction_submission_physics_residual.csv`: physics-informed residual candidate.

The final recommendation is:

```text
Forward: hybrid_w_0.65
Inverse: current design_submission.csv
```

Reason:

- The test set uses unseen gravity levels.
- Random KFold CV was not enough to evaluate OOD gravity behavior.
- Leave-One-Gravity-Out CV showed that the hybrid model is the strongest candidate for distance targets.
- The inverse-design file remained feasible under base, hybrid, alpha25, and alpha40 checks.

## 3. Load candidate files

In [2]:
target_cols = ["P80", "fines_frac", "oversize_frac", "R95", "R50_fines", "R50_oversize"]
distance_cols = ["R95", "R50_fines", "R50_oversize"]

expected_prediction_cols = ["scenario_id"] + target_cols
expected_design_cols = [
    "submission_id", "energy", "angle_rad", "coupling", "strength",
    "porosity", "gravity", "atmosphere", "shape_factor",
]

candidate_paths = {
    "base_extratrees": SUBMISSIONS_DIR / "prediction_submission.csv",
    "hybrid_w_0.65": SUBMISSIONS_DIR / "prediction_submission_hybrid_mlp_distance_w_0.65.csv",
    "alpha25_blend": SUBMISSIONS_DIR / "prediction_submission_controlled_blend_alpha_0.25.csv",
    "physics_residual": SUBMISSIONS_DIR / "prediction_submission_physics_residual.csv",
}

available_candidates = {}

for name, path in candidate_paths.items():
    if path.exists():
        available_candidates[name] = pd.read_csv(path)
        print(f"Loaded {name}: {path}")
    else:
        print(f"Missing {name}: {path}")

design_path = SUBMISSIONS_DIR / "design_submission.csv"
if not design_path.exists():
    raise FileNotFoundError(f"Missing inverse-design file: {design_path}")

design_submission = pd.read_csv(design_path)
print("Loaded design submission:", design_path)

Loaded base_extratrees: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission.csv
Missing hybrid_w_0.65: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_hybrid_mlp_distance_w_0.65.csv
Missing alpha25_blend: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_controlled_blend_alpha_0.25.csv
Missing physics_residual: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_physics_residual.csv
Loaded design submission: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/design_submission.csv


## 4. Submission format validation

In [3]:
def validate_prediction_submission(df, name):
    assert df.shape == (492, 7), f"{name}: expected shape (492, 7), got {df.shape}"
    assert df.columns.tolist() == expected_prediction_cols, f"{name}: wrong columns"
    assert df["scenario_id"].tolist() == list(range(492)), f"{name}: scenario_id must be 0..491"
    assert np.isfinite(df.drop(columns=["scenario_id"]).values).all(), f"{name}: contains non-finite values"

    assert (df[["P80", "R95", "R50_fines", "R50_oversize"]] >= 0).all().all(), (
        f"{name}: negative size or distance value"
    )

    assert ((df[["fines_frac", "oversize_frac"]] >= 0) &
            (df[["fines_frac", "oversize_frac"]] <= 1)).all().all(), (
        f"{name}: fraction outside [0, 1]"
    )

    print(f"{name}: prediction format OK")


def validate_design_submission(df):
    assert df.shape == (20, 9), f"design_submission: expected shape (20, 9), got {df.shape}"
    assert df.columns.tolist() == expected_design_cols, "design_submission: wrong columns"
    assert df["submission_id"].tolist() == list(range(20)), "design_submission: submission_id must be 0..19"
    assert np.isfinite(df.drop(columns=["submission_id"]).values).all(), "design_submission: contains non-finite values"
    print("design_submission: format OK")


for name, df in available_candidates.items():
    validate_prediction_submission(df, name)

validate_design_submission(design_submission)

base_extratrees: prediction format OK
design_submission: format OK


## 5. Candidate distribution comparison

The hybrid model is expected to produce larger distance predictions than the ExtraTrees baseline.

This is not automatically a problem because the test set contains unseen gravity values.
The purpose of this table is to check whether the final candidate remains within a plausible numerical range.

In [4]:
summary_rows = []

for name, df in available_candidates.items():
    row = {"candidate": name}

    for col in distance_cols:
        row[f"{col}_mean"] = df[col].mean()
        row[f"{col}_median"] = df[col].median()
        row[f"{col}_p75"] = df[col].quantile(0.75)
        row[f"{col}_p95"] = df[col].quantile(0.95)
        row[f"{col}_max"] = df[col].max()

    summary_rows.append(row)

candidate_distribution = pd.DataFrame(summary_rows)
display(candidate_distribution)

candidate_distribution.to_csv(
    REPORTS_DIR / "17_final_candidate_distribution_comparison.csv",
    index=False,
)

,candidate,R95_mean,R95_median,R95_p75,R95_p95,R95_max,R50_fines_mean,R50_fines_median,R50_fines_p75,R50_fines_p95,R50_fines_max,R50_oversize_mean,R50_oversize_median,R50_oversize_p75,R50_oversize_p95,R50_oversize_max
0,base_extratrees,367.20599,144.903024,589.193077,1235.550442,1654.971472,393.49665,156.017613,681.769264,1240.745131,1598.016175,178.261386,71.683292,328.076953,517.474989,677.693027


## 6. Load Leave-One-Gravity-Out results from notebook 16

The final decision is mainly based on notebook 16.

Expected files:

```text
reports/16_logo_summary_by_strategy.csv
reports/16_logo_metrics_by_strategy_target.csv
reports/16_final_decision_baseline_vs_alpha25.csv
```

In [5]:
logo_summary_path = REPORTS_DIR / "16_logo_summary_by_strategy.csv"
logo_metrics_path = REPORTS_DIR / "16_logo_metrics_by_strategy_target.csv"
decision_alpha25_path = REPORTS_DIR / "16_final_decision_baseline_vs_alpha25.csv"

if logo_summary_path.exists():
    logo_summary = pd.read_csv(logo_summary_path)
    display(logo_summary.sort_values("mean_norm_MAE"))
else:
    print("Missing:", logo_summary_path)

if logo_metrics_path.exists():
    logo_metrics = pd.read_csv(logo_metrics_path)
    display(
        logo_metrics
        .sort_values(["target", "normalized_MAE"])
        [["target", "strategy", "MAE", "RMSE", "R2", "P95_AE", "Bias", "normalized_MAE"]]
    )
else:
    print("Missing:", logo_metrics_path)

if decision_alpha25_path.exists():
    alpha25_decision = pd.read_csv(decision_alpha25_path)
    display(alpha25_decision)
else:
    print("Missing:", decision_alpha25_path)

,strategy,mean_norm_MAE,mean_R2,mean_P95_AE,mean_abs_bias
0,hybrid_w_0.65,0.221285,0.859072,111.932172,6.712336
1,physics_residual,0.261173,0.775831,148.296118,19.438114
2,alpha25,0.392738,0.615040,176.524145,8.890867
3,base,0.453561,0.492291,201.447550,9.617044


,target,strategy,MAE,RMSE,R2,P95_AE,Bias,normalized_MAE
0,P80,base,7.841240,10.349198,0.975220,21.382564,0.437683,0.119249
6,P80,hybrid_w_0.65,7.841240,10.349198,0.975220,21.382564,0.437683,0.119249
12,P80,alpha25,7.841240,10.349198,0.975220,21.382564,0.437683,0.119249
18,P80,physics_residual,7.841240,10.349198,0.975220,21.382564,0.437683,0.119249
10,R50_fines,hybrid_w_0.65,85.758140,122.130095,0.755231,277.177188,-18.566182,0.347341
22,R50_fines,physics_residual,121.285667,173.513075,0.505945,393.643866,-51.888721,0.491237
16,R50_fines,alpha25,188.334070,223.560726,0.179834,444.724439,-22.595067,0.762799
4,R50_fines,base,224.926193,261.725704,-0.124096,508.029820,-23.938028,0.911006
11,R50_oversize,hybrid_w_0.65,41.672160,58.567386,0.708187,123.138779,-3.474324,0.384298
23,R50_oversize,physics_residual,47.787989,73.614215,0.538983,167.797385,-22.860046,0.440698


,target,base_MAE_LOGO,alpha25_MAE_LOGO,alpha25_minus_base_MAE,paired_diff_ci_low,paired_diff_ci_high,prob_alpha25_better
0,R95,167.620273,143.084808,-24.535465,-25.127307,-23.986907,1.0
1,R50_fines,224.926193,188.334070,-36.592123,-37.320483,-35.913002,1.0
2,R50_oversize,88.650806,76.302547,-12.348259,-12.657293,-12.046057,1.0


## 7. Final interpretation

The key interpretation is:

```text
Random KFold CV made the baseline and hybrid look similar.
Leave-One-Gravity-Out CV showed that the hybrid is much stronger under gravity OOD validation.
```

This means the main failure mode is likely under-extrapolation on distance targets.

The hybrid model is selected because it reduces this OOD underprediction more effectively than the baseline and the alpha25 blend.

## 8. Optional: set the final forward file

In [6]:
# Safety switch:
# Set this to True only when you are ready to make the hybrid candidate
# the official forward prediction file.
CREATE_FINAL_FORWARD_FILE = True

selected_candidate_name = "hybrid_w_0.65"
selected_candidate_path = candidate_paths[selected_candidate_name]
official_prediction_path = SUBMISSIONS_DIR / "prediction_submission.csv"

if CREATE_FINAL_FORWARD_FILE:
    if not selected_candidate_path.exists():
        raise FileNotFoundError(f"Missing selected candidate: {selected_candidate_path}")

    shutil.copyfile(selected_candidate_path, official_prediction_path)

    print("Copied selected candidate to official prediction file.")
    print("Source:", selected_candidate_path)
    print("Destination:", official_prediction_path)

    final_pred = pd.read_csv(official_prediction_path)
    validate_prediction_submission(final_pred, "official prediction_submission.csv")
else:
    print("CREATE_FINAL_FORWARD_FILE = False")
    print("No file was overwritten.")
    print("To finalize manually, run:")
    print(f"cp {selected_candidate_path} {official_prediction_path}")

FileNotFoundError: Missing selected candidate: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission_hybrid_mlp_distance_w_0.65.csv

## 9. Final full validation

In [ ]:
pred = pd.read_csv(SUBMISSIONS_DIR / "prediction_submission.csv")
design = pd.read_csv(SUBMISSIONS_DIR / "design_submission.csv")

validate_prediction_submission(pred, "prediction_submission.csv")
validate_design_submission(design)

print("\nFinal submission files are valid.")
print("Prediction file:", SUBMISSIONS_DIR / "prediction_submission.csv")
print("Design file:", SUBMISSIONS_DIR / "design_submission.csv")

display(pred.head())
display(design.head())

prediction_submission.csv: prediction format OK
design_submission: format OK

Final submission files are valid.
Prediction file: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/prediction_submission.csv
Design file: /home/alouiyaz/projects/boom-challenge-ejecta-prediction/outputs/submissions/design_submission.csv


,scenario_id,P80,fines_frac,oversize_frac,R95,R50_fines,R50_oversize
0,0,120.852192,0.120046,0.208373,1064.238127,1012.805657,448.529033
1,1,113.668032,0.048219,0.158274,180.340190,191.185816,88.486206
2,2,149.740914,0.121343,0.384067,1306.856464,1316.118523,541.277606
3,3,162.227452,0.008732,0.543184,731.377793,891.581710,418.480874
4,4,137.978607,0.047156,0.357337,152.512030,142.451662,60.233150


,submission_id,energy,angle_rad,coupling,strength,porosity,gravity,atmosphere,shape_factor
0,0,3.598016,0.797647,1.291533,2.024534,0.304489,9.851536,0.422767,1.303159
1,1,3.219857,0.653336,0.937745,1.371766,0.268351,10.031512,0.819009,0.775107
2,2,3.883410,0.550208,0.754591,1.428060,0.294039,9.893283,0.800587,0.798041
3,3,2.769840,0.856038,0.933836,1.186456,0.266355,9.856101,0.429999,1.214199
4,4,3.491508,0.545414,1.620353,2.249113,0.264882,9.960773,0.715197,0.816463


## 10. Final summary

Final selected files:

```text
outputs/submissions/prediction_submission.csv
outputs/submissions/design_submission.csv
```

Recommended final forward source:

```text
outputs/submissions/prediction_submission_hybrid_mlp_distance_w_0.65.csv
```

Recommended final design source:

```text
outputs/submissions/design_submission.csv
```

Final rationale:

- The forward test set is OOD in gravity.
- Leave-One-Gravity-Out validation is a stronger proxy than random KFold for this problem.
- The hybrid MLP-distance model achieved the best LOGO performance on distance targets.
- The inverse-design solution remained feasible under base, hybrid, alpha25, and alpha40 rechecks.
- The physics-residual model was rejected because it was less stable and produced overly extreme test predictions.